## Problem 4.14 — Strawberry Jam

Strawberries contain about **15 wt% solids** and **85 wt% water**. To make strawberry jam, crushed strawberries and sugar are mixed in a **45:55 mass ratio**, and the mixture is heated to evaporate water until the residue contains **one-third water by mass**.

**(a)** Draw and label a flowchart of this process.

**(b)** Do the degree-of-freedom analysis and show that the system has zero degrees of freedom (i.e., the number of unknown process variables equals the number of equations relating them). If you have too many unknowns, think about what you might have forgotten to do.

**(c)** Calculate how many pounds of strawberries are needed to make a pound of jam.

**(d)** Making a pound of jam is something you could accomplish in your own kitchen (or maybe even a dorm room). However, a typical manufacturing line for jam might produce **1500 lbm/h**. List technical and economic factors you would have to take into account as you scaled up this process from your kitchen to a commercial operation.

![problem4.14](../artifacts/strawberry_4_14_env.png)

4 variables unknown: $m_M, m_W, m_G, m_A$

2 components $\to$ water and solids = 2 independent material balances

1 independent relationship: $\frac{m_M}{m_A}=\frac{45}{55}=k$

1 degree of freedom

- Hypotheses:
  - no accumulation
  - no reaction inside the heater

material balance for water: $0.85m_M = m_W + 1/3m_G$

material balance for solids: $0.15m_M + m_A = 2/3 m_G$

we know that: $m_M = km_A$

$$0.85(km_A) = m_W + 1/3m_G$$

$$0.15(km_A) + m_A = 2/3 m_G$$

suppose we want to calculate the flows based on $m_G$, we can reorganize this system into:

$$0.85k m_A-m_W=1/3m_G$$

$$(0.15k+1)m_A=2/3 m_G$$

or, in linear system notation:

\begin{bmatrix}0.85k & 1 \\ (0.15k+1) & 0\end{bmatrix}\begin{bmatrix}m_A \\ m_W\end{bmatrix}=\begin{bmatrix}1/3m_G \\ 2/3m_G\end{bmatrix}

In [1]:
import numpy as np

def calculate_stream_based_on_jam_mass(
        m_jam: float,
        k: float = 45./55.,
        strawberry_water_content: float = 0.85,
        jam_water_content: float = 1/3
) -> dict:
    """
    performs the material balance of jam heater to calculate all streams

    Args:
        m_jam (float): desired mass of jam produced
        k (float, optional): strawberry to sugar ratio. Defaults to 45./55..
        strawberry_water_content (float, optional): strawberry water conten. Defaults to 0.85.
        jam_water_content (float, optional): jam water content. Defaults to 1/3.

    Returns:
        dict: material balance metadata
    """
    strawberry_solids = 1 - strawberry_water_content

    # build the linear system
    A = np.array([
        [strawberry_water_content * k, -1],
        [strawberry_solids * (k + 1), 0]
    ])

    b = np.array([
        jam_water_content * m_jam,
        (1 - jam_water_content) * m_jam
    ])

    # solve the linear system
    sugar_mass, water_mass = np.linalg.solve(A, b)

    strawberry_mass = sugar_mass * k

    return {
        "strawberry_mass": strawberry_mass,
        "sugar_mass": sugar_mass,
        "water_mass": water_mass,
        "jam_mass": m_jam
    }

In [2]:
# Example usage
desired_jam_mass = 1000  # kg
result = calculate_stream_based_on_jam_mass(desired_jam_mass)
print(f"To produce {desired_jam_mass} kg of jam, you need:")
print(f"- {result['strawberry_mass']:.2f} kg of strawberries")
print(f"- {result['sugar_mass']:.2f} kg of sugar")
print(f"- {result['water_mass']:.2f} kg of water evaporated")

To produce 1000 kg of jam, you need:
- 2000.00 kg of strawberries
- 2444.44 kg of sugar
- 1366.67 kg of water evaporated


## Problem 4.6 — Partial Evaporation of an Acetone–Water Mixture

A liquid mixture of acetone and water contains **35 mole% acetone**. The mixture is to be partially evaporated to produce a vapor that is **75 mole% acetone** and leave a residual liquid that is **18.7 mole% acetone**.

### (a)

Suppose the process is to be carried out **continuously and at steady state** with a feed rate of **10.0 kmol/h**. Let $\dot{n}_v$ and $\dot{n}_l$ be the flow rates of the vapor and liquid product streams, respectively.

Draw and label a process flowchart, then write and solve balances on **total moles** and on **acetone** to determine the values of $\dot{n}_v$ and $\dot{n}_l$.

For each balance, state which terms in the general balance equation

$$
\text{accumulation}
=
\text{input}
+
\text{generation}
-
\text{output}
-
\text{consumption}
$$

can be discarded and why.

### (b)

Now suppose the process is to be carried out in a **closed container** that initially contains **10.0 kmol** of the liquid mixture. Let $n_v$ and $n_l$ be the moles of final vapor and liquid phases, respectively.

Draw and label a process flowchart, then write and solve **integral balances** on total moles and on acetone.

For each balance, state which terms of the general balance equation can be discarded and why.

### (c)

Returning to the continuous process, suppose the vaporization unit is built and started and the product stream flow rates and compositions are measured.

The measured acetone content of the vapor stream is **75 mole% acetone**, and the product stream flow rates have the values calculated in Part (a). However, the liquid product stream is found to contain **22.3 mole% acetone**.

It is possible that there is an error in the measured composition of the liquid stream, but give **at least five other reasons** for the discrepancy.

*Think about assumptions made in obtaining the solution of Part (a).*

In [3]:
# import the material balance module
from src.material_balances import StreamFactory

# Create a factory with shared component list
factory = StreamFactory(['Water', 'Acetone'], default_flow_type='molar')

# Create streams using positional compositions
feed = factory.add_stream('F-101', [0.65, 0.35], flow_rate=10)
vapor = factory.add_stream('V-101', [None, 0.75], direction='output')
liquid = factory.add_stream('L-101', [None, 0.187], direction='output')

# Build and solve the process unit
evaporator = factory.build_process_unit('H-101')
evaporator.solve_material_balances()
evaporator.print_report()




Process unit report: H-101
Mass balance tolerance: 1.0e-06

Input streams:
  F-101: total flow = 10 molar
    Acetone: fraction = 0.35, component flow = 3.5 molar
    Water: fraction = 0.65, component flow = 6.5 molar

Output streams:
  V-101: total flow = 2.8952 molar
    Acetone: fraction = 0.75, component flow = 2.1714 molar
    Water: fraction = 0.25, component flow = 0.723801 molar
  L-101: total flow = 7.1048 molar
    Acetone: fraction = 0.187, component flow = 1.3286 molar
    Water: fraction = 0.813, component flow = 5.7762 molar

Mass balance checks:
  Acetone: input = 3.5, output = 3.5, residual = -8.88178e-16 [PASS]
  Water: input = 6.5, output = 6.5, residual = 0 [PASS]
  Overall: input = 10, output = 10, residual = 0 [PASS]


In [5]:
# lets solve the jam production problem
factory = StreamFactory(
    component_names=['Solids','Water'],
    default_flow_type='kg/h'
)

# add streams
factory.add_stream('Strawberry', [0.15, 0.85])
factory.add_stream('Sugar', [1, 0])
factory.add_stream('Evaporated Water', [0., 1.], direction='output')
factory.add_stream('Jam', [2/3, 1/3], direction='output', flow_rate=1000)

# add ratio constraint
factory.add_ratio(stream1='Strawberry', stream2='Sugar', target_ratio=45/55)

# build process unit
evaporator = factory.build_process_unit('Jam Evaporator')

# calculate material balances
try:
    evaporator.solve_material_balances()
    evaporator.print_report()
except:
    evaporator.report_degrees_of_freedom()
    evaporator.suggest_missing_information()



Process unit report: Jam Evaporator
Mass balance tolerance: 1.0e-06

Input streams:
  Strawberry: total flow = 485.83 kg/h
    Solids: fraction = 0.15, component flow = 72.8745 kg/h
    Water: fraction = 0.85, component flow = 412.955 kg/h
  Sugar: total flow = 593.792 kg/h
    Solids: fraction = 1, component flow = 593.792 kg/h
    Water: fraction = 0, component flow = 0 kg/h

Output streams:
  Evaporated Water: total flow = 79.6221 kg/h
    Solids: fraction = 0, component flow = 0 kg/h
    Water: fraction = 1, component flow = 79.6221 kg/h
  Jam: total flow = 1000 kg/h
    Solids: fraction = 0.666667, component flow = 666.667 kg/h
    Water: fraction = 0.333333, component flow = 333.333 kg/h

Mass balance checks:
  Solids: input = 666.667, output = 666.667, residual = 2.27374e-13 [PASS]
  Water: input = 412.955, output = 412.955, residual = 7.61702e-12 [PASS]
  Overall: input = 1079.62, output = 1079.62, residual = 7.7307e-12 [PASS]


## Problem 4.13 — Checking the Consistency of Stream Analyses

A process is carried out in which a mixture containing **25.0 wt% methanol**, **42.5 wt% ethanol**, and the balance **water** is separated into two fractions.

A technician draws and analyzes samples of both product streams and reports that:

- one stream contains **39.8 wt% methanol** and **31.5 wt% ethanol**
- the other stream contains **19.7 wt% methanol** and **41.2 wt% ethanol**

You examine the reported figures and tell the technician that they must be wrong and that the stream analyses should be carried out again.

In [8]:
# check mass balances - assuming 1 kg/h of input
factory = StreamFactory(
    component_names=['Methanol','Ethanol', 'Water'],
    default_flow_type='kg/h'
)

# add streams
factory.add_stream('F-101', [0.25, 0.425, None], flow_rate=1)
factory.add_stream('O-101', [0.398, 0.315, None], direction='output')
factory.add_stream('O-102', [0.197, 0.412, None], direction='output')

# create equipment
mixer = factory.build_process_unit('MX-101')

# calculate material balances
try:
    mixer.report_degrees_of_freedom()
    mixer.solve_material_balances()
    mixer.print_report()
except:
    mixer.report_degrees_of_freedom()
    mixer.suggest_missing_information()



Degrees of Freedom Analysis for 'MX-101':
  Unknowns (flow rates + compositions): 2
  Independent material balance equations: 3
  Ratio constraints: 0
  Degrees of freedom: -1
  Status: SOLVABLE (need to provide 1 more value)

Process unit report: MX-101
Mass balance tolerance: 1.0e-06

Input streams:
  F-101: total flow = 1 kg/h
    Ethanol: fraction = 0.425, component flow = 0.425 kg/h
    Methanol: fraction = 0.25, component flow = 0.25 kg/h
    Water: fraction = 0.325, component flow = 0.325 kg/h

Output streams:
  O-101: total flow = 0.27015 kg/h
    Ethanol: fraction = 0.315, component flow = 0.0850972 kg/h
    Methanol: fraction = 0.398, component flow = 0.10752 kg/h
    Water: fraction = 0.287, component flow = 0.077533 kg/h
  O-102: total flow = 0.732826 kg/h
    Ethanol: fraction = 0.412, component flow = 0.301924 kg/h
    Methanol: fraction = 0.197, component flow = 0.144367 kg/h
    Water: fraction = 0.391, component flow = 0.286535 kg/h

Mass balance checks:
  Ethanol: in